# Frozen CLIP baseline on UCF101 (XOVABench), Kaggle

Runs Baseline A (frozen OpenAI CLIP, ViT-B/32 and ViT-B/16) on the XOV-Action UCF101 test list
(3,783 videos, full 101-class vocabulary, familiar/unfamiliar reported separately), plus temporal
and degradation diagnostics on the UCF101 development pool (official trainlist01, disjoint from
the test list).

**Before running**

1. Settings → Accelerator: **GPU** (T4 or P100). Settings → Internet: **On**.
2. Add input: dataset **lifailifai/ucf101** (full UCF101: 13,320 `.avi` under `UCF-101/`).
3. Run all cells. Outputs go to `/kaggle/working`; download `ovar_outputs.zip` at the end.

Rough time on a T4 session (decoding is CPU-bound): setup about 5 min, each full UCF101 pass
30–60 min, diagnostics about 10 min. The second backbone re-decodes the videos (cache keys
include the model); prompt-ensemble runs reuse the cached video embeddings.

Synthetic fixtures (smoke test) only check the software; they never enter the results report.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path

REPO = "eylulpelinkilic/generalized-category-discovery"
REF = "main"            # replace with a commit SHA to pin the code version
WORK = Path("/kaggle/working")
CODE = WORK / "generalized-category-discovery"
PROJECT = CODE / "action_recognition"


def run(cmd, cwd=None, allow_fail=False):
    """Run a shell command, stream its output, raise with the exit code on failure."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (PROJECT if PROJECT.exists() else WORK),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
    if code and not allow_fail:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")
    return code

## 1. Code: clone the repository

In [ ]:
if subprocess.run(["git", "ls-remote", f"https://github.com/{REPO}", REF],
                  capture_output=True, text=True).returncode != 0:
    raise RuntimeError("Cannot reach GitHub. Turn Settings -> Internet on (phone-verified account), "
                       "then restart the session.")

if not CODE.exists():
    CODE.mkdir(parents=True)
    run("git init -q && git remote add origin https://github.com/" + REPO + ".git", cwd=CODE)
run(f"git fetch -q --depth 1 origin {REF} && git checkout -q FETCH_HEAD", cwd=CODE)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True).stdout.strip()
print("code commit:", COMMIT)

## 2. Environment

In [ ]:
run("pip install -q open_clip_torch ftfy regex")
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode != 0:
    run("apt-get -qq update && apt-get -qq install -y ffmpeg")
run("ffmpeg -version | head -1")

import torch, open_clip
print("python", sys.version.split()[0], "| torch", torch.__version__, "| open_clip", open_clip.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Settings -> Accelerator: GPU"

In [ ]:
# Official repositories at pinned commits (XOV-Action label files, Open-VCLIP and TC-CLIP sources
# used by the tests), then the unit tests.
run("python -m ovar fetch-repos")
run("python -m unittest discover -s tests -t .")

## 3. Data paths

In [ ]:
candidates = sorted((p.parent for pattern in ("*/UCF-101/ApplyEyeMakeup", "*/*/UCF-101/ApplyEyeMakeup",
                                              "*/*/*/UCF-101/ApplyEyeMakeup")
                     for p in Path("/kaggle/input").glob(pattern)), key=lambda p: len(p.parts))
assert candidates, "UCF101 not found: add the dataset lifailifai/ucf101 as an input"
UCF_ROOT = candidates[0]
print("UCF101 root:", UCF_ROOT, "| other copies:", candidates[1:])

(PROJECT / "configs" / "paths.yaml").write_text(
    "dataset_roots:\n"
    f"  ucf101: {UCF_ROOT}\n"
    "hmdb51_single_class_folder: false\n"
    "checkpoints: {}\n"
    "cache_dir: cache\n"
    "runs_dir: runs\n")
print((PROJECT / "configs" / "paths.yaml").read_text())

## 4. Validate the attached copy

Every listed file must exist; a sample must decode. Inference also logs any decode failure per
video (failures stay in the denominator).

In [ ]:
run("python -m ovar validate --datasets ucf101 --check-files --check-decode --decode-limit 300")
run("python -m ovar validate --datasets ucf101 --split dev_train1 --check-files")

## 5. Weights and smoke test (synthetic fixtures, software check only)

In [ ]:
run("python -m ovar fetch-clip --arch ViT-B-32 --yes")
run("python -m ovar fetch-clip --arch ViT-B-16 --yes")
run("python -m ovar smoke-test")

## 6. Small-subset run (2 videos per class, seeded) before the full list

In [ ]:
run("python -m ovar infer clip_vitb32__common_16x1x1 --per-class 2 --set 'datasets=[ucf101]' "
    "--variant ucf101_subset2 --n-boot 200")

## 7. Full UCF101 test list (3,783 videos)

Common protocol: 16 segment-midpoint frames, 1 view, CLIP centre crop, prompt
"a video of a person {}.". The 28-template ensemble is a separate configuration.

In [ ]:
for exp in ["clip_vitb32__common_16x1x1", "clip_vitb32__common_16x1x1_ens28",
            "clip_vitb16__common_16x1x1", "clip_vitb16__common_16x1x1_ens28"]:
    run(f"python -m ovar infer {exp} --set 'datasets=[ucf101]' --variant ucf101")

## 8. Diagnostics on the development pool (not the test list)

In [ ]:
for diag in ["clip_vitb32__common_16x1x1__ucf_dev", "clip_vitb16__common_16x1x1__ucf_dev"]:
    run(f"python -m ovar diagnose {diag}")

## 9. Reports and outputs

In [ ]:
run("python -m ovar report")
for arch in ["vitb32", "vitb16"]:
    run(f"python -m ovar error-analysis runs/clip_{arch}__common_16x1x1__ucf101 "
        f"--diagnostics runs/diagnostics/clip_{arch}__common_16x1x1__ucf_dev")

from IPython.display import Markdown, display
display(Markdown((PROJECT / "reports" / "results.md").read_text()))

In [ ]:
(PROJECT / "runs" / "CODE_COMMIT.txt").write_text(COMMIT + "\n")
run(f"cd {PROJECT} && zip -qr {WORK}/ovar_outputs.zip runs reports configs/paths.yaml "
    "-x 'runs/smoke/fixtures/*'")
run(f"ls -lh {WORK}/ovar_outputs.zip")